# Contract-driven charging protocol

An executable prototype of [PROTOCOL.md](../docs/PROTOCOL.md). One charging
point carries $job:X\to\mathbb N_0$, where $0$ means unallocated. A vehicle asks
for a new job $k>0$ over a duration $d>0$, starting anywhere in a bounded start
set $I$:

$$
C^{\mathrm{req}}=\left\langle\top\;\middle|\;\Diamond_I\Box_d(job=k)\right\rangle.
$$

The protocol turns that into concrete offer contracts
$\left\langle\top\mid\Box_{[s,s+d)}(job=k)\right\rangle$, negotiates a price for
one of them, and stores it as a new commitment.

Every scheduling step is a contract operation — projection, strengthening,
weakening, merging, separation, and a consistency query — so the feasible
starts are read out of the repaired contract rather than computed beside it.
Time and prices use exact rational arithmetic, and every contract is evaluated
at anchor $0$ over $[0,\infty)$.

$I$ may be a finite union of intervals and singletons: a singleton is a discrete
start, a continuous component denotes every start in it. Following
PROTOCOL.md's grouping convention, a disconnected region $S$ appears as
$\Diamond_S\varphi=\bigvee_\ell\Diamond_{S_\ell}\varphi$ and
$\Box_S\varphi=\bigwedge_\ell\Box_{S_\ell}\varphi$ over its connected
components. This extends PROTOCOL.md's integer start grid without new syntax.

Run the cells in order with the project environment, from the repository root
or `notebooks/`. All state is in memory; these helpers are a prototype, not a
package API.

In [11]:
from dataclasses import dataclass, field
from fractions import Fraction as F
from itertools import combinations
from pathlib import Path
import sys

import portion as P
import interval_logic as il
from IPython.display import Markdown, display

notebook_dir = Path.cwd() if Path('steps.py').is_file() else Path.cwd() / 'notebooks'
if str(notebook_dir) not in sys.path:
    sys.path.insert(0, str(notebook_dir))
from steps import reset_step, step, println

job = il.Signal('job')
p = il.Proposition('p')


def fmt(value):
    if isinstance(value, P.Interval):
        return ' ∪ '.join(il.format_temporal_parameter(il.AbsoluteParameter(c))
                          for c in value) or '∅'
    if isinstance(value, tuple):
        return '(' + ', '.join(fmt(v) for v in value) + ')'
    return str(value)


def table(headers, rows):
    def line(values):
        return '| ' + ' | '.join(fmt(v).replace('|', r'\|') for v in values) + ' |'
    display(Markdown('\n'.join([line(headers), line(['---'] * len(headers)),
                                *(line(row) for row in rows)])))


def grouped(operator, body, region):
    '''Spell a region as one condition per connected component.'''
    connective = il.And if operator is il.Always else il.Or
    return connective(operator(body, component) for component in region).simplify(anchor=0)

## Contracts

A contract $\langle A\mid G\rangle$ is stored saturated, with $G$ replaced by
$G\lor\neg A$. Merging conjoins assumptions and guarantees; separation returns
the least contract that, merged with the other one, still refines this one.
`require_assumption` promotes an assumption to an explicit requirement, which is
how [PROTOCOL.md §3](../docs/PROTOCOL.md#3-repair-availability) forms
$\bar C^{\mathrm{post}}$.

Consistency asks whether some trajectory satisfies the whole guarantee. Here it
is `il.simplify(anchor=0)` failing to reach $\bot$. That is exact for the
reservation model — conflicting coverage of one signal is reduced to $\bot$ —
but in general an unreduced formula is evidence of nothing, so a richer model
needs a trajectory solver rather than this query.

`normalize` is the notebook's Boolean bookkeeping: conjunctive normal form
closed under resolution, with tautologies and subsumed clauses dropped, treating
temporal conditions as propositional leaves. It is exact but can blow up, and it
exists only to put the separation–merge result back into a readable shape. The
two temporal region rewrites this protocol needs — rewriting $\Box_I\Box_dp$ as
coverage of $I\oplus[0,d)$, and removing blocked times from a witness region —
now live in `il.simplify`; see
[CODE.md](../docs/CODE.md#parsing-formatting-rewriting-and-traversal).

In [12]:
@dataclass(frozen=True)
class Contract:
    assumption: il.Formula
    guarantee: il.Formula

    def __post_init__(self):
        a = il.to_nnf(self.assumption)
        object.__setattr__(self, 'assumption', a)
        object.__setattr__(self, 'guarantee', il.to_nnf(il.as_formula(self.guarantee) | ~a))

    def __str__(self):
        return f'⟨ {self.assumption} ∣ {self.guarantee} ⟩'

    def merge(self, other):
        a = self.assumption & other.assumption
        return Contract(a, (self.guarantee & other.guarantee) | ~a)

    def separate(self, other):
        a, g, b, h = self.assumption, self.guarantee, other.assumption, other.guarantee
        return Contract((a & h) | ~(g & b), g & b)

    def require_assumption(self):
        return Contract(True, self.assumption & self.guarantee)

    def normalize(self):
        return Contract(normalize(self.assumption), normalize(self.guarantee))


def consistent(contract):
    return contract.guarantee.simplify(anchor=0) != il.Boolean(False)


def _dual(literal):
    return il.to_nnf(~literal)


def _minimal_clauses(clauses):
    clauses = {c for c in clauses if not any(_dual(lit) in c for lit in c)}
    if frozenset() in clauses:
        return {frozenset()}
    return {c for c in clauses if not any(other < c for other in clauses)}


def _cnf(formula):
    # Temporal conditions are propositional leaves.
    if isinstance(formula, il.Boolean):
        return set() if formula.value else {frozenset()}
    if isinstance(formula, il.And):
        return _minimal_clauses(set().union(*(_cnf(c) for c in formula.operands)))
    if isinstance(formula, il.Or):
        result = {frozenset()}
        for child in formula.operands:
            result = _minimal_clauses({a | b for a in result for b in _cnf(child)})
        return result
    return {frozenset((formula,))}


def normalize(formula):
    '''Resolve and subsume to a fixed point, then simplify the regions.'''
    current = il.to_nnf(formula)
    while True:
        clauses = _cnf(current)
        while True:
            additions = set()
            for left, right in combinations(clauses, 2):
                for literal in left:
                    complement = _dual(literal)
                    if complement in right:
                        additions.add((left - {literal}) | (right - {complement}))
            reduced = _minimal_clauses(clauses | additions)
            if reduced == clauses:
                break
            clauses = reduced
        ordered = sorted(clauses, key=lambda c: tuple(sorted(map(str, c))))
        result = il.And(il.Or(sorted(c, key=str)) for c in ordered).simplify(anchor=0)
        if result == current:
            return result
        current = result

## Requests and commitments

A `Reservation` is a stored allocation: a positive job on one half-open
interval, which is the form
[PROTOCOL.md §1](../docs/PROTOCOL.md#1-project-stored-reservations-to-the-requested-job)
requires so that a later request can project it to unavailability. A `Request`
carries its job, duration, and start region. Each builds its contract once; the
protocol below then works on those contracts, and keeps the plain fields for the
concrete interval arithmetic of step 4.

In [13]:
@dataclass(frozen=True)
class Reservation:
    job: int
    interval: P.Interval
    contract: Contract = field(init=False)

    def __post_init__(self):
        if type(self.job) is not int or self.job <= 0:
            raise ValueError('a reservation needs a positive job id')
        if not self.interval.atomic or self.interval.empty:
            raise ValueError('a reservation is one nonempty half-open interval')
        object.__setattr__(self, 'contract',
                           Contract(True, il.Always(job.equals(self.job), self.interval)))

    @property
    def start(self):
        return self.interval.lower

    @property
    def duration(self):
        return self.interval.upper - self.interval.lower


@dataclass(frozen=True)
class Request:
    job: int
    duration: F
    starts: P.Interval
    contract: Contract = field(init=False)

    def __post_init__(self):
        if type(self.job) is not int or self.job <= 0:
            raise ValueError('a request needs a positive job id')
        if self.duration <= 0:
            raise ValueError('duration must be positive')
        if any(component.upper == P.inf for component in self.starts):
            raise ValueError('the start region must be bounded')
        object.__setattr__(self, 'contract', Contract(True, grouped(
            il.Eventually, il.Always(job.equals(self.job), self.duration), self.starts)))


def commitments(*reservations):
    contract = Contract(True, True)
    for reservation in reservations:
        contract = contract.merge(reservation.contract)
    return contract

## Steps 1–3: project, strengthen, repair

**[Project](../docs/PROTOCOL.md#1-project-stored-reservations-to-the-requested-job).**
Replace each $job=n$ atom by $p$ when $n$ is the requested job and $\neg p$
otherwise. Since the job is new, every commitment becomes $\Box_{R_\ell}\neg p$.
This forgets which other job holds the point, so the original commitments are
kept for step 4.

**[Strengthen](../docs/PROTOCOL.md#2-conjunctively-strengthen-the-request-and-test-contract-consistency).**
Turn each existential start alternative $\Diamond_{I_\ell}\Box_dp$ into the
universal requirement $\Box_{I_\ell}\Box_dp$, which `il.simplify` rewrites as
coverage of the envelope $I\oplus[0,d)$. Merged with the projected commitments,
an inconsistent candidate exposes a conflict anywhere in that envelope — it does
not yet mean the request is infeasible.

**[Repair](../docs/PROTOCOL.md#3-repair-availability).**
Weaken coverage to the existence of an available time, separate against the
projected commitments, merge back, and require the resulting assumption. The
repaired guarantee keeps the commitments alongside a witness region that
`il.simplify` has already stripped of the blocked times. `availability` reads
that region back out — it is the output of the repair, not a separate
calculation. When the candidate was consistent there is nothing to repair and
the weakened envelope is used directly.

`analyze` returns only the availability contract; everything else is logged.

In [14]:
def project(contract, target_job):
    literal = il.NaturalLiteral(target_job)

    def visit(node):
        if isinstance(node, il.Equal):
            return p if literal in (node.left, node.right) else ~p
        children = tuple(visit(child) for child in node.children)
        return node._with_children(children) if children != node.children else node

    return Contract(visit(contract.assumption), visit(contract.guarantee))


def _terms(formula, node_type):
    return formula.operands if isinstance(formula, node_type) else (formula,)


def strengthen(request_contract):
    return Contract(True, il.And(
        il.Always(alternative.body, alternative.parameter)
        for alternative in _terms(request_contract.guarantee, il.Or)
    ))


def weaken(coverage_contract):
    return Contract(True, il.Or(
        il.Eventually(condition.body, condition.parameter)
        for condition in _terms(normalize(coverage_contract.guarantee), il.And)
    ))


def availability(contract):
    '''The witness region the contract still allows the requested job.'''
    region = P.empty()
    for term in _terms(normalize(contract.guarantee), il.And):
        for alternative in _terms(term, il.Or):
            if isinstance(alternative, il.Eventually) and alternative.body == p:
                region |= alternative.parameter.interval
    return region


def analyze(request, commitment):
    with step('Project the request and the commitments'):
        cmt = project(commitment, request.job)
        println(f'{cmt = !s}', '')
        req = project(request.contract, request.job)
        println(f'{req = !s}')

    with step('Strengthen and decide consistency'):
        strong = strengthen(req).normalize()
        println(f'{strong = !s}', '')
        candidate = cmt.merge(strong)
        println(f'{candidate = !s}', '')
        passes = consistent(candidate)
        println(f'{passes = !s}')

    with step('Derive the availability contract'):
        weak = weaken(strong)
        println(f'{weak = !s}', '')
        if passes:
            avail = weak
        else:
            with step('Separate, merge, and require the assumption'):
                separated = weak.separate(cmt)
                println(f'{separated = !s}', '')
                post = separated.merge(cmt).require_assumption().normalize()
                println(f'{post = !s}')
            avail = Contract(True, grouped(il.Eventually, p, availability(post)))
        println(f'{avail = !s}')
    return avail

## Step 4: concrete offers

[PROTOCOL.md §4](../docs/PROTOCOL.md#4-concretize-feasible-intervals) enumerates
the starts that keep the full duration inside the available region $J$:

$$
F=I\cap\{s:[s,s+d)\subseteq J\},
\qquad
\mathcal J=\{[s,s+d):s\in F\}.
$$

`erode` is that containment test. Its upper bound is closed at $b-d$, because a
window may end exactly where an excluded right endpoint begins, and a component
shorter than $d$ contributes no start at all.

$F$ is the whole feasible region, so representatives are chosen from it rather
than from a grid: every singleton survives, and a continuous component
contributes its left endpoint when included and otherwise its midpoint. No
nonempty component can therefore be missed. Each offer is checked against the
request and against the original commitments — the job vocabulary, not the
projection.

In [15]:
def erode(region, duration):
    '''Starts whose full half-open window fits inside the region.'''
    starts = P.empty()
    for component in region:
        if component.upper - duration >= component.lower:
            starts |= P.Interval.from_atomic(
                component.left, component.lower, component.upper - duration, P.CLOSED)
    return starts


def refines(offer, request):
    return (offer.job == request.job
            and offer.duration == request.duration
            and not (request.starts & erode(offer.interval, request.duration)).empty)


def synthesize(request, avail, commitment):
    with step('Compute the feasible starts'):
        feasible = request.starts & erode(availability(avail), request.duration)
        println(f'feasible = {fmt(feasible)}')

    with step('Generate the offer contracts'):
        offers = []
        for component in feasible:
            start = (component.lower if component.left == P.CLOSED
                     else (component.lower + component.upper) / 2)
            offer = Reservation(request.job, P.closedopen(start, start + request.duration))
            assert refines(offer, request)
            assert consistent(commitment.merge(offer.contract))
            println(f'{offer.contract!s}')
            offers.append(offer)
    return tuple(offers)

In [16]:
reset_step()
C_cmt = commitments(Reservation(1, P.closedopen(F(2), F(3))))
request = Request(2, F(1), P.closedopen(F(1), F(5)))
C_avail = analyze(request, C_cmt)
offers = synthesize(request, C_avail, C_cmt)
table(['Offer contract', 'Refines the request', 'Consistent with commitments'],
      [(offer.contract, refines(offer, request), consistent(C_cmt.merge(offer.contract)))
       for offer in offers])

⏺ Step 1: Project the request and the commitments
│ cmt = ⟨ ⊤ ∣ □_[2,3) ¬p ⟩
│                         
│ req = ⟨ ⊤ ∣ ◇_[1,5) □_1 p ⟩
╯
⏺ Step 2: Strengthen and decide consistency
│ strong = ⟨ ⊤ ∣ □_[1,6) p ⟩
│                           
│ candidate = ⟨ ⊤ ∣ □_[1,6) p ∧ □_[2,3) ¬p ⟩
│                                           
│ passes = False
╯
⏺ Step 3: Derive the availability contract
│ weak = ⟨ ⊤ ∣ ◇_[1,6) p ⟩
│                         
├ ⏺ Step 3.1: Separate, merge, and require the assumption
│ │ separated = ⟨ □_[1,6) ¬p ∨ □_[2,3) ¬p ∣ ◇_[1,6) p ⟩
│ │                                                    
│ │ post = ⟨ ⊤ ∣ (◇_[1,2) p ∨ ◇_[3,6) p) ∧ □_[2,3) ¬p ⟩
├─╯ 
│ avail = ⟨ ⊤ ∣ ◇_[1,2) p ∨ ◇_[3,6) p ⟩
╯
⏺ Step 4: Compute the feasible starts
│ feasible = [1,1] ∪ [3,5)
╯
⏺ Step 5: Generate the offer contracts
│ ⟨ ⊤ ∣ □_[1,2) job = 2 ⟩
│ ⟨ ⊤ ∣ □_[3,4) job = 2 ⟩
╯


| Offer contract | Refines the request | Consistent with commitments |
| --- | --- | --- |
| ⟨ ⊤ ∣ □_[1,2) job = 2 ⟩ | True | True |
| ⟨ ⊤ ∣ □_[3,4) job = 2 ⟩ | True | True |

## Step 5: negotiate a price

[PROTOCOL.md §5](../docs/PROTOCOL.md#5-negotiate-interval-price-pairs) keeps the
offers fixed and bargains only over the total price $q$, with
$u^{\mathrm{EV}}=v(J)-q$ and $u^{\mathrm{CH}}=q-c(J)$. The charger proposes
first, each rejection switches roles, and targets concede by
$t_{r+1}=t_r(1-r/T)$ with $\lambda=1$, so round 0 leaves them unchanged.
Before the deadline the evaluator needs both its target and its floor; at the
deadline it needs only to exceed its floor.

The values below are PROTOCOL.md's illustration: costs $2$ and $3$ from tariffs
of $0.20$ and $0.30$ per kWh on $10$ kWh, EV values $8$ and $7$ from an
alternative costing $8$ with inconvenience $0$ and $1$, floors $1$, initial
targets $(6,4)$, and deadline $T=2$.

In [17]:
def utility(party, offer, price, costs, values):
    return values[offer] - price if party == 'EV' else price - costs[offer]


def bargain(offers, costs, values, targets, floors, deadline=2):
    '''Alternating offers over fixed contracts, in exact arithmetic.'''
    offers = tuple(sorted(offers, key=lambda offer: offer.start))
    targets, history, proposer = dict(targets), [], 'CH'
    for r in range(deadline + 1):
        evaluator = 'EV' if proposer == 'CH' else 'CH'
        priced = [(offer, costs[offer] + targets[proposer] if proposer == 'CH'
                   else values[offer] - targets[proposer]) for offer in offers]
        priced = [(offer, price) for offer, price in priced
                  if utility(proposer, offer, price, costs, values) >= floors[proposer]]
        row = dict(round=r, targets=dict(targets), proposer=proposer, priced=priced)
        if not priced:
            history.append(row | dict(utilities=(), decision='No admissible offers'))
            return None, None, history, 'no_admissible_offers'
        utilities = tuple(utility(evaluator, offer, price, costs, values)
                          for offer, price in priced)
        best, price = priced[max(range(len(priced)), key=lambda i: utilities[i])]
        top = max(utilities)
        accepted = (top > floors[evaluator] if r == deadline
                    else top >= targets[evaluator] and top >= floors[evaluator])
        history.append(row | dict(utilities=utilities,
                                  decision='Accept' if accepted else 'Reject'))
        if accepted:
            return best, price, history, 'accepted'
        if r == deadline:
            return None, None, history, 'deadline_rejection'
        targets = {party: target * (1 - F(r, deadline)) for party, target in targets.items()}
        proposer = evaluator

In [18]:
cost = dict(zip(offers, (F(2), F(3)), strict=True))
value = dict(zip(offers, (F(8), F(7)), strict=True))
floors = {'EV': F(1), 'CH': F(1)}
accepted, agreed_price, rounds, status = bargain(
    offers, cost, value, {'EV': F(6), 'CH': F(4)}, floors)
with step('Negotiate over the fixed offers'):
    println(f'Result: {status}', f'Accepted contract: {accepted.contract}')
    println(f'Total price: {agreed_price}')
table(['Round', 'Targets (EV, CH)', 'Proposer', 'Offers (interval, price)',
       'Evaluator utilities', 'Decision'],
      [(r['round'], (r['targets']['EV'], r['targets']['CH']), r['proposer'],
        '; '.join(f'({fmt(offer.interval)}, {price})' for offer, price in r['priced']),
        r['utilities'], r['decision']) for r in rounds])
assert (accepted, agreed_price) == (offers[0], F(4))

⏺ Step 6: Negotiate over the fixed offers
│ Result: accepted                          
│ Accepted contract: ⟨ ⊤ ∣ □_[1,2) job = 2 ⟩
│ Total price: 4
╯


| Round | Targets (EV, CH) | Proposer | Offers (interval, price) | Evaluator utilities | Decision |
| --- | --- | --- | --- | --- | --- |
| 0 | (6, 4) | CH | ([1,2), 6); ([3,4), 7) | (2, 0) | Reject |
| 1 | (6, 4) | EV | ([1,2), 2); ([3,4), 1) | (0, -2) | Reject |
| 2 | (3, 2) | CH | ([1,2), 4); ([3,4), 5) | (4, 2) | Accept |

## Step 6: validate and store

[PROTOCOL.md §6](../docs/PROTOCOL.md#6-validate-and-store-the-accepted-agreement)
revalidates the accepted contract against the current request and commitments
before inserting it, and stores the price as metadata beside the reservation.

This helper is sequential and in memory. A real service must validate and insert
atomically against one snapshot of the request, commitments, and model inputs,
restarting the scheduling when they have moved:

```text
snapshot := current request contract, commitment contract, and model inputs
offers := synthesize(analyze(snapshot))
if offers empty: end without agreement
agreement := bargain(offers)
if no agreement: end without agreement
atomically:
    if current inputs differ from snapshot: restart scheduling
    if the accepted contract fails refinement or consistency: restart scheduling
    merge the accepted contract into the commitments and store the price
```

A price rejection ends the run without agreement; only changed inputs force a
rescheduling. The strengthened coverage, the repaired availability, and the
unaccepted offers are all temporary.

In [19]:
def commit(commitment, request, offer, price):
    if offer is None:
        return 'no_agreement', commitment, None
    if not refines(offer, request) or not consistent(commitment.merge(offer.contract)):
        return 'restart', commitment, None
    return 'committed', commitment.merge(offer.contract).normalize(), (offer, price)

In [20]:
status, C_committed, record = commit(C_cmt, request, accepted, agreed_price)
with step('Revalidate and store the accepted contract'):
    println(f'Status: {status}', f'Updated commitment: {C_committed}')
    println(f"EV utility: {utility('EV', accepted, agreed_price, cost, value)}")
    println(f"Charger utility: {utility('CH', accepted, agreed_price, cost, value)}")
    println(f'Projection for a later job 3: {project(C_committed, 3).normalize()}')
stored, stored_price = record
table(['Stored job', 'Interval', 'Price'], [(stored.job, stored.interval, stored_price)])
assert status == 'committed'
assert (stored.job, stored.interval, stored_price) == (2, P.closedopen(1, 2), F(4))
assert utility('EV', accepted, agreed_price, cost, value) == 4
assert utility('CH', accepted, agreed_price, cost, value) == 2

⏺ Step 7: Revalidate and store the accepted contract
│ Status: committed                                            
│ Updated commitment: ⟨ ⊤ ∣ □_[1,2) job = 2 ∧ □_[2,3) job = 1 ⟩
│ EV utility: 4
│ Charger utility: 2
│ Projection for a later job 3: ⟨ ⊤ ∣ □_[1,3) ¬p ⟩
╯


| Stored job | Interval | Price |
| --- | --- | --- |
| 2 | [1,2) | 4 |